In [2]:
import os
import json
import pdal
from pathlib import Path

def traiter_dalles_locales_pdal(dossier_entree="data/brut_laz/", dossier_sortie="outputs/sol_laz/", emprise_bbox=None):
    """
    Parcourt les fichiers .laz/.copc.laz locaux, applique un filtre de sol PDAL 
    et enregistre les résultats filtrés.
    """
    # Création du dossier de sortie si besoin
    Path(dossier_sortie).mkdir(parents=True, exist_ok=True)
    
    # Recherche de tous les fichiers LiDAR dans le dossier d'entrée
    fichiers_laz = list(Path(dossier_entree).glob("*.la*"))
    
    if not fichiers_laz:
        print(f"⚠️ Aucun fichier LiDAR trouvé dans le dossier '{dossier_entree}'.")
        return []

    fichiers_traites = []
    
    for chemin_laz in fichiers_laz:
        nom_fichier = chemin_laz.name
        chemin_sortie = os.path.join(dossier_sortie, f"sol_{nom_fichier}")
        
        print(f"\n🔄 Traitement de la dalle : {nom_fichier}")
        
        # Construction dynamique du pipeline PDAL
        pipeline_stages = [
            {
                "type": "readers.las",
                "filename": str(chemin_laz)
            }
        ]
        
        # Si une emprise géographique (bounding box) est fournie, on recadre
        if emprise_bbox:
            xmin, ymin, xmax, ymax = emprise_bbox
            pipeline_stages.append({
                "type": "filters.crop",
                "bounds": f"([({xmin},{xmax}), ({ymin},{ymax})])"
            })
            
        # Algorithme de classification du sol (Simplemrf ou CSF)
        pipeline_stages.extend([
            {
                "type": "filters.classify",
                "algorithm": "simplemrf"
            },
            {
                "type": "filters.range",
                "limits": "Classification[2:2]"  # Code LAS standard 2 = Sol (Ground)
            },
            {
                "type": "writers.las",
                "filename": chemin_sortie,
                "compression": "true"
            }
        ])
        
        try:
            # Exécution du pipeline pour la dalle courante
            pipeline_json = json.dumps(pipeline_stages)
            pipeline = pdal.Pipeline(pipeline_json)
            nb_points = pipeline.execute()
            
            print(f"✅ Succès : {nb_points} points de sol extraits vers '{chemin_sortie}'")
            fichiers_traites.append(chemin_sortie)
            
        except Exception as e:
            print(f"❌ Erreur lors du traitement PDAL de {nom_fichier} : {e}")
            
    return fichiers_traites

# --- Exemple d'utilisation ---
# 1. Traiter toutes les dalles du dossier sans découpage supplémentaire :
dalles_sol = traiter_dalles_locales_pdal()
#
# 2. Ou en restreignant à une emprise précise (Lambert 93) autour de votre station WIM :
bbox_zone = (769000.0, 6895000.0, 770000.0, 6896000.0)
dalles_sol = traiter_dalles_locales_pdal(emprise_bbox=bbox_zone)

ModuleNotFoundError: No module named 'pdal'